#Evaluating LLMs for Semantic Risk Discovery in Corner Case Scenarios
###Jayaashri Chezhian

This notebook implements a controlled evaluation pipeline for multimodal LLMs (Gemini, ChatGPT, Llama) on autonomous driving corner-case scenarios.

It focuses on:

- semantic understanding (entities + relations)
- risk identification + severity scoring
- runtime performance across batch sizes

#Mount Google Drive
Mounting the drive allows the notebook to access files and folders stored in Drive, enabling reading of datasets and saving of outputs.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Define Directory Paths
Defines file paths for key directories.

* `dataset`: where the image dataset lives,
* `output`: where model raw outputs are saved,
* `results`: where processed experiment summaries are stored

In [2]:
dataset = "/content/drive/MyDrive/sementic web project/data/Dataset"
output = "/content/drive/MyDrive/semantic web project 2/model output"
results = "/content/drive/MyDrive/sementic web project 2/results"

#Library Imports

Imports core Python libraries:

* `os` for file handling,
* `random` for reproducible scenario sampling,
* `json` for saving structured outputs,
* `re` for regular-expression based parsing

In [3]:
import os
import random
import json
import re
import time
from PIL import Image
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 16.7 MB/s eta 0:00:00


#Data Exploration and Scenario Selection

A quick sanity check to confirm the dataset is mounted correctly and to inspect available weather/condition folders.

In [4]:
print("Dataset folders:")
print(os.listdir(dataset))

Dataset folders:
['DayFoggy', 'DayRainy', 'DayClear', 'NightRainy', 'NightClear', 'NightFoggy']


## Scenario Extraction Function

Parses dataset hierarchy to extract structured scenario metadata including condition, anomaly type, and scenario name.

It converts the raw folder structure into something the experiment can systematically sample from.

In [5]:
def get_scenarios(base_path):
    structure = {}

    for condition in sorted(os.listdir(base_path)):
        condition_path = os.path.join(base_path, condition)
        if not os.path.isdir(condition_path):
            continue

        structure[condition] = {}

        for anomaly in sorted(os.listdir(condition_path)):
            anomaly_path = os.path.join(condition_path, anomaly)
            if not os.path.isdir(anomaly_path):
                continue

            scenarios = [
                s for s in sorted(os.listdir(anomaly_path))
                if os.path.isdir(os.path.join(anomaly_path, s))
            ]

            structure[condition][anomaly] = scenarios

    return structure

data_structure = get_scenarios(dataset)

## Random Selection

Selects a subset of scenarios using a fixed random seed to ensure reproducibility.

In [6]:
random.seed(42)

selected_conditions = ["DayClear", "DayFoggy", "NightRainy"]
selected = []

for cond in selected_conditions:
    for anomaly in data_structure[cond]:
        available_scenarios = data_structure[cond][anomaly]

        if len(available_scenarios) == 0:
            continue

        scenario = random.choice(available_scenarios)

        selected.append({
            "condition": cond,
            "anomaly": anomaly,
            "scenario": scenario
        })

if len(selected) > 5:
    selected = random.sample(selected, 5)

print("Total selected scenarios:", len(selected))
for item in selected:
    print(item)

Total selected scenarios: 5
{'condition': 'NightRainy', 'anomaly': 'StateAnomaly', 'scenario': 'StaticHighwayAd4'}
{'condition': 'DayClear', 'anomaly': 'EvidenceBasedAnomaly', 'scenario': 'FallenTree'}
{'condition': 'DayFoggy', 'anomaly': 'EvidenceBasedAnomaly', 'scenario': 'FootballFollowedByBoy'}
{'condition': 'DayFoggy', 'anomaly': 'BehaviorAnomaly', 'scenario': 'CarOppositeDirectionHIT'}
{'condition': 'DayClear', 'anomaly': 'BehaviorAnomaly', 'scenario': 'BusObscuresCar'}


## Alternative Sampling for environmental conditions - Manual Scenario Selection



In [ ]:
#I used this for getting 5 scenario  for differnt weather conditions

# random.seed(39)

# selected_conditions = ["NightFoggy"]
# selected = []

# for cond in selected_conditions:
#     for anomaly in data_structure[cond]:
#         available_scenarios = data_structure[cond][anomaly]

#         if len(available_scenarios) == 0:
#             continue

#         scenario = random.choice(available_scenarios)

#         selected.append({
#             "condition": cond,
#             "anomaly": anomaly,
#             "scenario": scenario
#         })

# all_options = []
# for cond in selected_conditions:
#     for anomaly in data_structure[cond]:
#         for scenario in data_structure[cond][anomaly]:
#             all_options.append((cond, anomaly, scenario))

# while len(selected) < 5 and len(all_options) > 0:
#     cond, anomaly, scenario = random.choice(all_options)

#     # avoid duplicates
#     if not any(item["scenario"] == scenario for item in selected):
#         selected.append({
#             "condition": cond,
#             "anomaly": anomaly,
#             "scenario": scenario
#         })

# if len(selected) > 5:
#     selected = random.sample(selected, 5)

# # Step 4: Shuffle so no order bias
# random.shuffle(selected)

# print("Total selected scenarios:", len(selected))
# for item in selected:
#     print(item)

Total selected scenarios: 5
{'condition': 'NightFoggy', 'anomaly': 'BehaviorAnomaly', 'scenario': 'PoliceCarChase1'}
{'condition': 'NightFoggy', 'anomaly': 'BehaviorAnomaly', 'scenario': 'CarDropoff'}
{'condition': 'NightFoggy', 'anomaly': 'StateAnomaly', 'scenario': 'StaticHighwayAd2'}
{'condition': 'NightFoggy', 'anomaly': 'BehaviorAnomaly', 'scenario': 'BlinkingYellowIntersection'}
{'condition': 'NightFoggy', 'anomaly': 'EvidenceBasedAnomaly', 'scenario': 'FootballOnHighway'}


## Save Scenarios
Selected scenarios are saved to JSON

In [7]:
save_path = "/content/drive/MyDrive/sementic web project/results/selected_scenarios.json"

with open(save_path, "w") as f:
    json.dump(selected, f, indent=4)

print("Saved selected scenarios")

Saved selected scenarios


# Data Preparation

## Image Loading
Retrieves image paths corresponding to each selected scenario.

In [8]:
def load_images(scenario_info):
    cond = scenario_info["condition"]
    anomaly = scenario_info["anomaly"]
    scenario = scenario_info["scenario"]

    path = os.path.join(dataset, cond, anomaly, scenario)

    images = []
    for file in sorted(os.listdir(path)):
        if file.endswith((".png", ".jpg", ".jpeg")):
            img_path = os.path.join(path, file)
            images.append(img_path)

    return images

## Batch Formation
Groups images into fixed-size batches to evaluate model performance under varying input sizes.

In [9]:
def create_batches(images, batch_size):
    batches = []
    for i in range(0, len(images), batch_size):
        batches.append(images[i:i + batch_size])
    return batches

## Filename Sanitization
Cleans scenario names to ensure compatibility with file system naming constraints.

In [10]:
def safe_name(text):
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", text)


for scenario_info in selected:

    scenario_id = safe_name(
        f'{scenario_info["condition"]}_{scenario_info["anomaly"]}_{scenario_info["scenario"]}'
    )

    print("Scenario ID:", scenario_id)

Scenario ID: NightRainy_StateAnomaly_StaticHighwayAd4
Scenario ID: DayClear_EvidenceBasedAnomaly_FallenTree
Scenario ID: DayFoggy_EvidenceBasedAnomaly_FootballFollowedByBoy
Scenario ID: DayFoggy_BehaviorAnomaly_CarOppositeDirectionHIT
Scenario ID: DayClear_BehaviorAnomaly_BusObscuresCar


Verifies existence of output and result directories before execution.

In [11]:
print("Output exists:", os.path.exists(output))
print("Results exists:", os.path.exists(results))

Output exists: True
Results exists: False


## Output Normalization and Risk Extraction
Processes raw model outputs to extract structured risk descriptions and severity scores using regex-based parsing.

Converts unstructured LLM output into structured evaluable data.

In [12]:
def clean_text(s):
    if not s:
        return ""
    s = s.replace("**", "").replace("*", "")
    s = s.replace("\n", " ").replace("\r", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def extract_risks(text):
    risks = []


    blocks = re.split(r'(?=(?:\*\*)?Risk\s*\d+\s*:)', text, flags=re.IGNORECASE)

    for block in blocks:
        if not re.search(r'(?:\*\*)?Risk\s*\d+\s*:', block, flags=re.IGNORECASE):
            continue

        num_match = re.search(r'(?:\*\*)?Risk\s*(\d+)\s*:', block, flags=re.IGNORECASE)

        type_match = re.search(
            r'(?:Name|Risk Type)\s*:\s*(.*)',
            block,
            flags=re.IGNORECASE
        )

        desc_match = re.search(
            r'Description\*{0,2}\s*:\s*(.*?)(?=\n.*?(?:\*\*)?(?:Severity|Severity Score)\*{0,2}\s*:|$)',
            block,
            flags=re.IGNORECASE | re.DOTALL
        )

        sev_match = re.search(
            r'(?:\*\*)?(?:Severity|Severity Score)\*{0,2}\s*:?\s*\*{0,2}\s*([0-9]*\.?[0-9]+)',
            block,
            flags=re.IGNORECASE
        )

        if desc_match:
            risks.append({
                "risk_number": int(num_match.group(1)) if num_match else None,
                "risk_type": clean_text(type_match.group(1)) if type_match else "",
                "description": clean_text(desc_match.group(1)),
                "severity": float(sev_match.group(1)) if sev_match else None
            })

    if risks:
        return risks

    turtle_pattern = re.compile(
    r'([A-Za-z_][\w\-]*:[A-Za-z_][\w\-]*)\s+'
    r'(?:a|rdf:type)\s+([A-Za-z_][\w\-]*:[A-Za-z_][\w\-]*)\s*;\s*'
    r'.*?(?:hasDescription|description)\s+"(.*?)"\s*;\s*'
    r'.*?hasSeverityScore\s+([0-9]*\.?[0-9]+)',
    flags=re.IGNORECASE | re.DOTALL
    )

    for i, match in enumerate(turtle_pattern.finditer(text), start=1):
        risk_id = match.group(1)
        risk_type = match.group(2)
        description = match.group(3)
        severity = match.group(4)

        risks.append({
            "risk_number": i,
            "risk_type": clean_text(risk_type),
            "risk_id": clean_text(risk_id),
            "description": clean_text(description),
            "severity": float(severity)
        })

    return risks

##RDF Risk Extraction

 Parses Turtle (RDF) output to extract risk type, description, and severity.
 Identifies unique risk entities and converts them into structured format for evaluation.

In [13]:
from rdflib import Graph, RDF, URIRef, Literal

def extract_risks_from_rdf(ttl_text):
    risks = []

    if not ttl_text or ttl_text.strip() == "":
        return risks

    try:
        g = Graph()
        g.parse(data=ttl_text, format="turtle")

        seen_risk_ids = set()

        for subject in set(g.subjects()):
            subject_str = str(subject)

            if "risk" not in subject_str.lower():
                continue

            risk_id = subject_str.split("#")[-1].split("/")[-1]

            if risk_id in seen_risk_ids:
                continue

            seen_risk_ids.add(risk_id)

            risk_type = ""
            description = ""
            severity = None

            for obj in g.objects(subject, RDF.type):
                risk_type = str(obj).split("#")[-1].split("/")[-1]

            for pred, obj in g.predicate_objects(subject):
                pred_name = str(pred).split("#")[-1].split("/")[-1].lower()

                if "description" in pred_name:
                    description = str(obj)

                if "severity" in pred_name:
                    try:
                        severity = float(obj)
                    except:
                        severity = None

            risks.append({
                "risk_number": len(risks) + 1,
                "risk_type": risk_type,
                "risk_id": risk_id,
                "description": clean_text(description),
                "severity": severity
            })

        return risks

    except Exception as e:
        print("    RDF parsing failed:", e)
        return []

## RDF/Turtle Extraction
Extracts RDF triples from model outputs for semantic representation of scene understanding. Enables semantic graph analysis of model outputs.

In [14]:
def extract_ttl(text):
    match = re.search(r"```(?:turtle)?(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

# Prompt Engineering

Defines a standardized prompt requiring entity extraction, relationship identification, RDF generation, and risk assessment.

In [ ]:
prompt = """You are an autonomous driving safety analyst.
Your task is to analyze the following scene images and produce a semantic interpretation of the environment. The images represent a potential corner case scenario.
Please perform the following tasks:
1. Identify the main entities in the scene
(vehicles, pedestrians, cyclists, road infrastructure, obstacles, weather conditions).
2. Identify relationships between entities
(approaching, following, crossing, occluding, near).
3. Generate RDF triples describing the scene using Turtle format.
4. Identify potential safety risks present in the scene in RDF risk class.
5. Assign a severity score between 0 and 1 for each risk.
Return the results in the following format:
Scene Description in RDF Triples (Turtle Format)
Identified Risks for each risk class
Risk 1:
Description:
Severity Score:"""

## Experiment Setup
Prepares structured experiment data including scenarios, images, and batches based on selected batch size.

In [15]:
batch_size = int(input("Enter batch size (e.g., 1, 3, or 5): "))

all_scenarios_data = []

for scenario_info in selected:
    images = load_images(scenario_info)
    batches = create_batches(images, batch_size)

    scenario_record = {
        "condition": scenario_info["condition"],
        "anomaly": scenario_info["anomaly"],
        "scenario": scenario_info["scenario"],
        "images": images,
        "batches": batches
    }

    all_scenarios_data.append(scenario_record)

    print("\nScenario:", scenario_info["scenario"])
    print("Condition:", scenario_info["condition"])
    print("Anomaly:", scenario_info["anomaly"])
    print("Total images:", len(images))
    print("Total batches:", len(batches))

Enter batch size (e.g., 1, 3, or 5): 5

Scenario: StaticHighwayAd4
Condition: NightRainy
Anomaly: StateAnomaly
Total images: 10
Total batches: 2

Scenario: FallenTree
Condition: DayClear
Anomaly: EvidenceBasedAnomaly
Total images: 11
Total batches: 3

Scenario: FootballFollowedByBoy
Condition: DayFoggy
Anomaly: EvidenceBasedAnomaly
Total images: 11
Total batches: 3

Scenario: CarOppositeDirectionHIT
Condition: DayFoggy
Anomaly: BehaviorAnomaly
Total images: 10
Total batches: 2

Scenario: BusObscuresCar
Condition: DayClear
Anomaly: BehaviorAnomaly
Total images: 10
Total batches: 2


# Model Execution Pipelines


**Model Execution**

A unified inference pipeline is applied across all models, where batched multimodal inputs are processed using a consistent prompt, with runtime tracking and structured output storage ensuring comparable evaluation conditions.

**Output Aggregation**

Raw model outputs are normalized and parsed into structured representations, enabling systematic aggregation of risk and severity information into comparable evaluation reports.

## Gemini Model Inference
Processes each batch using Gemini API, records runtime, and stores outputs and RDF triples.

In [ ]:
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [ ]:
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 29.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.50.0 which is incompatible.


In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Say: Gemini is connected."
)

print(response.text)

Gemini is connected.


In [ ]:
model_name = "gemini-3-flash-preview"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

all_results = []

for scenario_data in all_scenarios_data:
    scenario_id = safe_name(
        f'{scenario_data["condition"]}_{scenario_data["anomaly"]}_{scenario_data["scenario"]}'
    )

    print(f"\nProcessing scenario: {scenario_id}")

    for batch_index, batch_paths in enumerate(scenario_data["batches"], start=1):
        print(f"  Batch {batch_index}/{len(scenario_data['batches'])}")

        batch_images = [Image.open(img_path) for img_path in batch_paths]

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                start_time = time.time()

                response = client.models.generate_content(
                    model="gemini-3-flash-preview",
                    contents=[prompt, *batch_images]
                )

                end_time = time.time()
                runtime = end_time - start_time

                raw_output = response.text
                ttl_output = extract_ttl(raw_output)

                output_data = {
                    "model": model_name,
                    "scenario": {
                        "model": model_name,
                        "condition": scenario_data["condition"],
                        "anomaly": scenario_data["anomaly"],
                        "scenario": scenario_data["scenario"]
                    },
                    "batch_size": batch_size,
                    "batch_number": batch_index,
                    "batch_paths": batch_paths,
                    "runtime_seconds": runtime,
                    "raw_output": raw_output,
                }

                json_file = os.path.join(output,f"{model_name}_bs{batch_size}_{scenario_id}_batch{batch_index}_output.json")
                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(output_data, f, indent=4)

                ttl_file = os.path.join(output, f"{model_name}_bs{batch_size}_{scenario_id}_batch{batch_index}_output.ttl")
                with open(ttl_file, "w", encoding="utf-8") as f:
                    f.write(ttl_output if ttl_output else "")

                all_results.append(output_data)

                print(f"    Runtime: {runtime:.2f} sec")
                print(f"    Saved: {json_file}")

                success = True
                time.sleep(2)

            except Exception as e:
                retries -= 1
                print(f"    Error: {e}")
                print("    Retrying...")
                time.sleep(5)

        if not success:
            print(f"    Failed batch {batch_index}, skipping...")


Processing scenario: NightRainy_StateAnomaly_StaticHighwayAd4
  Batch 1/2
    Runtime: 17.24 sec
    Saved: /content/drive/MyDrive/sementic web project/out/gemini-3-flash-preview_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
  Batch 2/2
    Runtime: 17.61 sec
    Saved: /content/drive/MyDrive/sementic web project/out/gemini-3-flash-preview_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.json

Processing scenario: DayClear_EvidenceBasedAnomaly_FallenTree
  Batch 1/3
    Runtime: 18.97 sec
    Saved: /content/drive/MyDrive/sementic web project/out/gemini-3-flash-preview_bs5_DayClear_EvidenceBasedAnomaly_FallenTree_batch1_output.json
  Batch 2/3
    Runtime: 18.27 sec
    Saved: /content/drive/MyDrive/sementic web project/out/gemini-3-flash-preview_bs5_DayClear_EvidenceBasedAnomaly_FallenTree_batch2_output.json
  Batch 3/3
    Runtime: 11.05 sec
    Saved: /content/drive/MyDrive/sementic web project/out/gemini-3-flash-preview_bs5_DayClear_EvidenceBasedAnomaly

### Gemini Output Aggregation

In [ ]:
output_txt_path = f"gemini3_report_batch{batch_size}.txt"

fixed_summary = []
target_model = "gemini-3-flash-preview"

with open(output_txt_path, "w", encoding="utf-8") as txt_file:
    for file_name in os.listdir(output):
        if not file_name.endswith(".json"):
            continue
        if not file_name.startswith(target_model):
            continue
        if f"_bs{batch_size}_" not in file_name:
            continue
        file_path = os.path.join(output, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        scenario = data["scenario"]
        runtime = data.get("runtime_seconds", 0)
        batch_number = data.get("batch_number")
        batch_paths = data.get("batch_paths", [])
        raw_output = data.get("raw_output", "")

        parsed_risks = extract_risks(raw_output)

        header = f"\n{'-'*70}\n"
        file_info = f"File: {file_name}\n"
        scenario_info = f"Scenario: {scenario['condition']} | {scenario['anomaly']} | {scenario['scenario']}\n"
        batch_info = f"Batch: {batch_number}\n"
        runtime_info = f"Runtime: {runtime}\n"
        size_info = f"Input size: {len(batch_paths)}\nOutput size: {len(parsed_risks)}\n"

        print(header + file_info + scenario_info + batch_info + runtime_info + size_info, end="")
        txt_file.write(header + file_info + scenario_info + batch_info + runtime_info + size_info)

        for i, risk in enumerate(parsed_risks, start=1):
            risk_text = f"  {i}. {risk['description']}\n     Severity: {risk['severity']}\n"
            print(risk_text, end="")
            txt_file.write(risk_text)

        for risk in parsed_risks:
            fixed_summary.append({
                "condition": scenario["condition"],
                "anomaly": scenario["anomaly"],
                "scenario": scenario["scenario"],
                "batch": batch_number,
                "risk_description": risk["description"],
                "severity": risk["severity"],
                "runtime": runtime,
                "input_size": len(batch_paths),
                "output_size": len(parsed_risks)
            })

try:
    from google.colab import files
    files.download(output_txt_path)
except ImportError:
    print(f"\nDone! Your file is saved at: {os.path.abspath(output_txt_path)}")


----------------------------------------------------------------------
File: gemini-3-flash-preview_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
Scenario: NightRainy | StateAnomaly | StaticHighwayAd4
Batch: 1
Runtime: 17.243394374847412
Input size: 5
Output size: 4
  1. Severely Reduced Visibility. The combination of nighttime and active rainfall significantly limits the effective range of cameras and LiDAR sensors. The dark environment, coupled with atmospheric interference from rain, makes it difficult to identify non-illuminated objects, pedestrians, or road debris.
     Severity: 0.85
  2. Reduced Road Friction (Hydroplaning Risk). The road surface is visibly wet and reflective. This condition reduces the coefficient of friction between the tires and the asphalt, resulting in increased braking distances and a higher risk of losing control during maneuvers at the intersection.
     Severity: 0.7
  3. Sensor Interference from Reflections. The wet pavement acts as 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ChatGPT Multimodal Inference

In [ ]:
!pip -q install -U openai

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.7 MB/s eta 0:00:00
Enter your OpenAI API key: ··········


In [ ]:
import base64
import mimetypes

def encode_image_to_data_url(image_path):
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"

In [ ]:
def call_chatgpt_batch(image_paths, prompt, model_name="gpt-5.4-mini"):
    content = [{"type": "input_text", "text": prompt}]

    for img_path in image_paths:
        mime_type, _ = mimetypes.guess_type(img_path)
        if mime_type is None:
            mime_type = "image/png"

        with open(img_path, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")

        data_url = f"data:{mime_type};base64,{encoded}"

        content.append({
            "type": "input_image",
            "image_url": data_url
        })

    response = openai_client.responses.create(
        model=model_name,
        input=[{
            "role": "user",
            "content": content
        }]
    )

    return response.output_text

In [ ]:
chatgpt_model_name = "gpt-5.4-mini"
all_chatgpt_results = []

for scenario_data in all_scenarios_data:
    scenario_id = safe_name(
        f'{scenario_data["condition"]}_{scenario_data["anomaly"]}_{scenario_data["scenario"]}'
    )

    print(f"\nProcessing ChatGPT scenario: {scenario_id}")

    for batch_index, batch_paths in enumerate(scenario_data["batches"], start=1):
        print(f" Batch {batch_index}/{len(scenario_data['batches'])}")

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                start_time = time.time()

                raw_output = call_chatgpt_batch(
                    image_paths=batch_paths,
                    prompt=prompt,
                    model_name=chatgpt_model_name
                )

                end_time = time.time()
                runtime = end_time - start_time

                ttl_output = extract_ttl(raw_output)

                output_data = {
                    "model": chatgpt_model_name,
                    "scenario": {
                        "model": chatgpt_model_name,
                        "condition": scenario_data["condition"],
                        "anomaly": scenario_data["anomaly"],
                        "scenario": scenario_data["scenario"]
                    },
                    "batch_size": batch_size,
                    "batch_number": batch_index,
                    "batch_paths": batch_paths,
                    "runtime_seconds": runtime,
                    "raw_output": raw_output
                }

                json_file = os.path.join(
                    output,
                    f"{chatgpt_model_name}_bs{batch_size}_{scenario_id}_batch{batch_index}_output.json"
                )

                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(output_data, f, indent=4)

                ttl_file = os.path.join(
                    output,
                    f"{chatgpt_model_name}_bs{batch_size}_{scenario_id}_batch{batch_index}_output.ttl"
                )

                with open(ttl_file, "w", encoding="utf-8") as f:
                    f.write(ttl_output if ttl_output else "")

                all_chatgpt_results.append(output_data)

                print(f" Runtime: {runtime:.2f} sec")
                print(f" Saved: {json_file}")

                success = True
                time.sleep(2)

            except Exception as e:
                retries -= 1
                print(f" Error: {e}")
                print(" Retrying...")
                time.sleep(5)

        if not success:
            print(f" Failed batch {batch_index}, skipping...")


Processing ChatGPT scenario: NightRainy_StateAnomaly_StaticHighwayAd4
 Batch 1/10
 Runtime: 6.21 sec
 Saved: /content/drive/MyDrive/sementic web project/output/gpt-5.4-mini_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
 Batch 2/10
 Runtime: 5.72 sec
 Saved: /content/drive/MyDrive/sementic web project/output/gpt-5.4-mini_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.json
 Batch 3/10
 Runtime: 7.23 sec
 Saved: /content/drive/MyDrive/sementic web project/output/gpt-5.4-mini_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch3_output.json
 Batch 4/10
 Runtime: 11.91 sec
 Saved: /content/drive/MyDrive/sementic web project/output/gpt-5.4-mini_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch4_output.json
 Batch 5/10
 Runtime: 7.20 sec
 Saved: /content/drive/MyDrive/sementic web project/output/gpt-5.4-mini_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch5_output.json
 Batch 6/10
 Runtime: 8.77 sec
 Saved: /content/drive/MyDrive/sementic web project/output/gpt

### ChatGPT Output Aggregation

In [ ]:
output_txt_path = f"chatgpt_report_batch{batch_size}.txt"

fixed_summary = []
target_model = "gpt-5.4-mini"

with open(output_txt_path, "w", encoding="utf-8") as txt_file:
    for file_name in os.listdir(output):
        if not file_name.endswith(".json"):
            continue
        if not file_name.startswith(target_model):
            continue
        if f"_bs{batch_size}_" not in file_name:
            continue
        file_path = os.path.join(output, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        scenario = data["scenario"]
        runtime = data.get("runtime_seconds", 0)
        batch_number = data.get("batch_number")
        batch_paths = data.get("batch_paths", [])
        raw_output = data.get("raw_output", "")

        parsed_risks = extract_risks(raw_output)

        header = f"\n{'-'*70}\n"
        file_info = f"File: {file_name}\n"
        scenario_info = f"Scenario: {scenario['condition']} | {scenario['anomaly']} | {scenario['scenario']}\n"
        batch_info = f"Batch: {batch_number}\n"
        runtime_info = f"Runtime: {runtime}\n"
        size_info = f"Input size: {len(batch_paths)}\nOutput size: {len(parsed_risks)}\n"

        print(header + file_info + scenario_info + batch_info + runtime_info + size_info, end="")
        txt_file.write(header + file_info + scenario_info + batch_info + runtime_info + size_info)

        for i, risk in enumerate(parsed_risks, start=1):
            risk_text = f"  {i}. {risk['description']}\n     Severity: {risk['severity']}\n"
            print(risk_text, end="")
            txt_file.write(risk_text)

        for risk in parsed_risks:
            fixed_summary.append({
                "condition": scenario["condition"],
                "anomaly": scenario["anomaly"],
                "scenario": scenario["scenario"],
                "batch": batch_number,
                "risk_description": risk["description"],
                "severity": risk["severity"],
                "runtime": runtime,
                "input_size": len(batch_paths),
                "output_size": len(parsed_risks)
            })

try:
    from google.colab import files
    files.download(output_txt_path)
except ImportError:
    print(f"\n file is saved at: {os.path.abspath(output_txt_path)}")


----------------------------------------------------------------------
File: gpt-5.4-mini_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
Scenario: NightRainy | StateAnomaly | StaticHighwayAd4
Batch: 1
Runtime: 6.20766282081604
Input size: 1
Output size: 5
  1. Reduced visibility due to nighttime conditions and rain may limit perception of lane boundaries, infrastructure, and distant hazards.
     Severity: 0.78
  2. Wet road surface may reduce tire traction and increase braking distance, raising the risk of skidding or loss of control.
     Severity: 0.72
  3. Overpass, poles, and signal structures create visual clutter and may occlude parts of the roadway or reduce detection of critical cues.
     Severity: 0.43
  4. Traffic signal state is unclear in the scene, which may lead to ambiguous right-of-way interpretation at the intersection ahead.
     Severity: 0.36
  5. The road appears empty, which can still be risky in low-light rain because other road users may be h

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Llama (Groq) Inference

In [ ]:
!pip -q install groq pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.3 MB/s eta 0:00:00


In [ ]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
def encode_image_for_groq(path):
    mime_type, _ = mimetypes.guess_type(path)
    if mime_type is None:
        mime_type = "image/png"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"

In [ ]:
def call_llama_batch(image_paths, prompt, model_name):
    def encode(path):
        mime, _ = mimetypes.guess_type(path)
        if mime is None:
            mime = "image/png"
        with open(path, "rb") as f:
            return f"data:{mime};base64," + base64.b64encode(f.read()).decode()

    content = [{"type": "text", "text": prompt}]

    for p in image_paths:
        content.append({
            "type": "image_url",
            "image_url": {"url": encode(p)}
        })

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": content}],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
llama_model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
all_llama_results = []

for scenario_data in all_scenarios_data:
    scenario_id = safe_name(
        f'{scenario_data["condition"]}_{scenario_data["anomaly"]}_{scenario_data["scenario"]}'
    )

    print(f"\nProcessing Llama scenario: {scenario_id}")

    for batch_index, batch_paths in enumerate(scenario_data["batches"], start=1):
        print(f" Batch {batch_index}/{len(scenario_data['batches'])}")

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                start_time = time.time()

                raw_output = call_llama_batch(
                    image_paths=batch_paths,
                    prompt=prompt,
                    model_name=llama_model_name
                )

                end_time = time.time()
                runtime = end_time - start_time

                ttl_output = extract_ttl(raw_output)

                output_data = {
                    "model": llama_model_name,
                    "scenario": {
                        "model": llama_model_name,
                        "condition": scenario_data["condition"],
                        "anomaly": scenario_data["anomaly"],
                        "scenario": scenario_data["scenario"]
                    },
                    "batch_size": batch_size,
                    "batch_number": batch_index,
                    "batch_paths": batch_paths,
                    "runtime_seconds": runtime,
                    "raw_output": raw_output
                }

                safe_model_name = llama_model_name.replace("/", "-")

                json_file = os.path.join(
                    output,
                    f"{safe_model_name}_bs{batch_size}_{scenario_id}_batch{batch_index}_output.json"
                )

                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(output_data, f, indent=4)

                ttl_file = os.path.join(
                    output,
                    f"{safe_model_name}_bs{batch_size}_{scenario_id}_batch{batch_index}_output.ttl"
                )

                with open(ttl_file, "w", encoding="utf-8") as f:
                    f.write(ttl_output if ttl_output else "")

                all_llama_results.append(output_data)

                print(f" Runtime: {runtime:.2f} sec")
                print(f" Saved: {json_file}")

                success = True
                time.sleep(2)

            except Exception as e:
                retries -= 1
                print(f" Error: {e}")
                print(" Retrying...")
                time.sleep(5)

        if not success:
            print(f" Failed batch {batch_index}, skipping...")


Processing Llama scenario: NightRainy_StateAnomaly_StaticHighwayAd4
 Batch 1/10
 Runtime: 3.14 sec
 Saved: /content/drive/MyDrive/sementic web project/output/meta-llama-llama-4-scout-17b-16e-instruct_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
 Batch 2/10
 Runtime: 3.42 sec
 Saved: /content/drive/MyDrive/sementic web project/output/meta-llama-llama-4-scout-17b-16e-instruct_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.json
 Batch 3/10
 Runtime: 3.38 sec
 Saved: /content/drive/MyDrive/sementic web project/output/meta-llama-llama-4-scout-17b-16e-instruct_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch3_output.json
 Batch 4/10
 Runtime: 3.56 sec
 Saved: /content/drive/MyDrive/sementic web project/output/meta-llama-llama-4-scout-17b-16e-instruct_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch4_output.json
 Batch 5/10
 Runtime: 3.55 sec
 Saved: /content/drive/MyDrive/sementic web project/output/meta-llama-llama-4-scout-17b-16e-instruct_bs1_NightRainy_

### Llama (Groq) Output Aggregation

In [ ]:
output_txt_path = f"llama_groq_report_batch{batch_size}.txt"

fixed_summary = []
target_model = "meta-llama-llama-4-scout-17b-16e-instruct"

with open(output_txt_path, "w", encoding="utf-8") as txt_file:
    for file_name in os.listdir(output):
        if not file_name.endswith(".json"):
            continue
        if not file_name.startswith(target_model):
            continue
        if f"_bs{batch_size}_" not in file_name:
            continue

        file_path = os.path.join(output, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        scenario = data["scenario"]
        runtime = data.get("runtime_seconds", 0)
        batch_number = data.get("batch_number")
        batch_paths = data.get("batch_paths", [])
        raw_output = data.get("raw_output", "")

        parsed_risks = extract_risks(raw_output)

        header = f"\n{'-'*70}\n"
        file_info = f"File: {file_name}\n"
        scenario_info = f"Scenario: {scenario['condition']} | {scenario['anomaly']} | {scenario['scenario']}\n"
        batch_info = f"Batch: {batch_number}\n"
        runtime_info = f"Runtime: {runtime}\n"
        size_info = f"Input size: {len(batch_paths)}\nOutput size: {len(parsed_risks)}\n"

        print(header + file_info + scenario_info + batch_info + runtime_info + size_info, end="")
        txt_file.write(header + file_info + scenario_info + batch_info + runtime_info + size_info)

        for i, risk in enumerate(parsed_risks, start=1):
            risk_text = f"  {i}. {risk['description']}\n     Severity: {risk['severity']}\n"
            print(risk_text, end="")
            txt_file.write(risk_text)

        for risk in parsed_risks:
            fixed_summary.append({
                "model": target_model,
                "condition": scenario["condition"],
                "anomaly": scenario["anomaly"],
                "scenario": scenario["scenario"],
                "batch": batch_number,
                "risk_description": risk["description"],
                "severity": risk["severity"],
                "runtime": runtime,
                "input_size": len(batch_paths),
                "output_size": len(parsed_risks)
            })

try:
    from google.colab import files
    files.download(output_txt_path)
except ImportError:
    print(f"\nfile is saved at: {os.path.abspath(output_txt_path)}")


----------------------------------------------------------------------
File: meta-llama-llama-4-scout-17b-16e-instruct_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
Scenario: NightRainy | StateAnomaly | StaticHighwayAd4
Batch: 1
Runtime: 3.141613245010376
Input size: 1
Output size: 5
  1. The scene is dark and rainy, significantly reducing visibility. The streetlights, although present, may not provide sufficient illumination to reveal all obstacles or entities on the road in a timely manner.
     Severity: 0.8
  2. The road appears wet due to rain, which can make the road surface slippery. This condition can lead to reduced traction for the vehicle, increasing the risk of skidding or losing control.
     Severity: 0.7
  3. The darkness and rain may obscure obstacles or entities on the road, such as pedestrians, cyclists, or other vehicles. The reduced visibility increases the risk of not detecting these obstacles in time to react.
     Severity: 0.9
  4. The combina

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Experimental Pipeline - baseline

Dataset → Scenario Selection → Batching → Model Inference → Parsing → Reporting

#Ontology Based Pipeline

In [16]:
!pip install rdflib

#Ontology Loading
Loads the ontology file in Turtle (TTL) format using rdflib.
Parses it into a graph structure and prints the total number of triples to confirm successful loading.

In [17]:
from rdflib import Graph, RDF, RDFS, OWL

ontology_path = "/content/drive/MyDrive/semantic web project 2/ontology ttl file/PRO2.ttl"

g = Graph()
g.parse(ontology_path, format="turtle")

print("Ontology loaded.")
print("Total triples:", len(g))


Ontology loaded.
Total triples: 306


#Ontology Parsing
Parses the ontology file and extracts key components such as classes, subclass relationships, object properties, and datatype properties. It also retrieves labels, comments, and domain, range information, and summarizes the ontology by counting labels, comments, and total triples for further use in prompting and analysis.

In [18]:
def short_name(uri):
    uri = str(uri)
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.rstrip("/").split("/")[-1]

def get_label(graph, uri):
    label = graph.value(uri, RDFS.label)
    return str(label) if label else short_name(uri)

def get_comment(graph, uri):
    comment = graph.value(uri, RDFS.comment)
    return str(comment) if comment else ""

def parse_ontology(ontology_path):
    g = Graph()
    g.parse(ontology_path, format="turtle")

    classes = []
    object_properties = []
    datatype_properties = []
    subclass_relations = []

    for cls in g.subjects(RDF.type, OWL.Class):
        classes.append({
            "name": short_name(cls),
            "label": get_label(g, cls),
            "comment": get_comment(g, cls)
        })

    for child, parent in g.subject_objects(RDFS.subClassOf):
        subclass_relations.append({
            "child": short_name(child),
            "parent": short_name(parent)
        })

    for prop in g.subjects(RDF.type, OWL.ObjectProperty):
      domain = g.value(prop, RDFS.domain)
      range_ = g.value(prop, RDFS.range)

      label = get_label(g, prop)
      comment = get_comment(g, prop)

      object_properties.append({
        "name": short_name(prop),
        "label": label,
        "comment": comment if comment else "No description provided",
        "domain": short_name(domain) if domain else "Unknown",
        "range": short_name(range_) if range_ else "Unknown"
    })


    for prop in g.subjects(RDF.type, OWL.DatatypeProperty):
      domain = g.value(prop, RDFS.domain)
      range_ = g.value(prop, RDFS.range)

      label = get_label(g, prop)
      comment = get_comment(g, prop)

      datatype_properties.append({
        "name": short_name(prop),
        "label": label,
        "comment": comment if comment else "No description provided",
        "domain": short_name(domain) if domain else "Unknown",
        "range": short_name(range_) if range_ else "Unknown"
    })

    num_labels = len(list(g.triples((None, RDFS.label, None))))
    num_comments = len(list(g.triples((None, RDFS.comment, None))))

    return {
        "classes": classes,
        "subclass_relations": subclass_relations,
        "object_properties": object_properties,
        "datatype_properties": datatype_properties,
        "num_labels": num_labels,
        "num_comments": num_comments,
        "total_triples": len(g)
    }

#Ontology Summary Statistics
Displays the extracted ontology statistics, including the number of classes, subclass relationships, object properties, datatype properties, labels, and comments. This provides a quick overview of the ontology structure and confirms successful parsing.

In [19]:
ontology_data = parse_ontology(ontology_path)

print("Classes:", len(ontology_data["classes"]))
print("Subclass relations:", len(ontology_data["subclass_relations"]))
print("Object properties:", len(ontology_data["object_properties"]))
print("Datatype properties:", len(ontology_data["datatype_properties"]))
print("Labels:", ontology_data["num_labels"])
print("Comments:", ontology_data["num_comments"])

Classes: 79
Subclass relations: 72
Object properties: 12
Datatype properties: 3
Labels: 54
Comments: 54


In [20]:

print("\nCLASSES")
for c in ontology_data["classes"]:
    print(f"- {c['name']}: {c['comment']}")




CLASSES
- Actor: A dynamic entity in the scene such as a vehicle or pedestrian.
- Advertisement: A visual display such as a billboard that may distract or be misinterpreted as a traffic sign.
- Ambulance: 
- AnimalNearRoadIndicator: Presence of animals near the road indicating unpredictable movement and potential collision risk.
- Barrel: 
- BehaviorAnomaly: A situation where an actor behaves in an unexpected or unsafe way.
- BrokenLine: 
- Bus: 
- Car: 
- ClearCondition: Weather condition with clear visibility and no significant environmental disturbances.
- CollisionRisk: Risk of a collision between the vehicle and another object or actor.
- ConstructionBarricade: 
- ConstructionWorker: 
- ConstructionZoneIndicator: Presence of construction activity indicating potential hazards such as workers or blocked lanes.
- CornerCase: A rare or unusual driving scenario that may pose safety risks.
- Crosswalk: 
- Cyclist: 
- DayCondition: Driving conditions during daylight with generally good 

In [21]:
print("\nSUBCLASS HIERARCHY")
for s in ontology_data["subclass_relations"]:
    print(f"- {s['child']} subclassOf {s['parent']}")




SUBCLASS HIERARCHY
- Advertisement subclassOf RoadObject
- LaneMarking subclassOf RoadObject
- Obstacle subclassOf RoadObject
- OccludingObject subclassOf RoadObject
- TrafficSign subclassOf RoadObject
- TrafficSignal subclassOf RoadObject
- Ambulance subclassOf EmergencyVehicle
- FireTruck subclassOf EmergencyVehicle
- PoliceCar subclassOf EmergencyVehicle
- AnimalNearRoadIndicator subclassOf EvidenceIndicator
- ConstructionZoneIndicator subclassOf EvidenceIndicator
- FallingObjectIndicator subclassOf EvidenceIndicator
- OccludedPedestrianIndicator subclassOf EvidenceIndicator
- RollingBallIndicator subclassOf EvidenceIndicator
- Barrel subclassOf Obstacle
- ConstructionBarricade subclassOf Obstacle
- Dog subclassOf Obstacle
- FallenTree subclassOf Obstacle
- FootBall subclassOf Obstacle
- HayBale subclassOf Obstacle
- TrafficCone subclassOf Obstacle
- VendingMachine subclassOf Obstacle
- ZooAnimal subclassOf Obstacle
- BehaviorAnomaly subclassOf CornerCase
- EvidenceBasedAnomaly sub

In [22]:
print("\nOBJECT PROPERTIES")
for p in ontology_data["object_properties"]:
    print(f"- {p['label']} ({p['domain']} → {p['range']})")
    print(f"  Comment: {p['comment']}")



OBJECT PROPERTIES
- has actor (Scene → Actor)
  Comment: Links a scene to the actors present in it.
- has corner case type (CornerCase → CornerCase)
  Comment: Specifies the type of anomaly associated with a corner case.
- has evidence (CornerCase → EvidenceIndicator)
  Comment: Links a corner case to observable evidence indicating potential risks.
- has obstacle (Scene → Obstacle)
  Comment: Links a scene to obstacles present in it.
- has pedestrian (Scene → Pedestrian)
  Comment: Links a scene to pedestrians present in it.
- has risk (CornerCase → Risk)
  Comment: Links a corner case scenario to the risks identified in it.
- has road object (Scene → RoadObject)
  Comment: Links a scene to road objects present in it.
- has scene type (Scene → Scene)
  Comment: Describes the type of environment where the scene takes place.
- has vehicle (Scene → Vehicle)
  Comment: Links a scene to vehicles present in it.
- has weather condition (Scene → WeatherCondition)
  Comment: Describes the envi

In [23]:
print("\nDATATYPE PROPERTIES")
for p in ontology_data["datatype_properties"]:
    print(f"- {p['label']} ({p['domain']} → {p['range']})")
    print(f"  Comment: {p['comment']}")


DATATYPE PROPERTIES
- has description (CornerCase → string)
  Comment: Provides a textual description of a scenario or identified risk.
- has scenario name (CornerCase → string)
  Comment: Stores the name or identifier of the corner case scenario
- has severity score (Risk → float)
  Comment: Assigns a numerical severity score to a risk indicating its intensity.


#Ontology Prompt Builder
Creates a summarized version of the ontology for use in the ontology-based prompt. It organizes corner case categories, main scene concepts, risk concepts, evidence indicators, object/road concepts, allowed properties, and datatype properties into a readable format.

#Allowed Classes and Relationships Builder
Generates lists of valid ontology classes, relationships, and class descriptions. These are inserted into the prompt to guide the model toward consistent RDF output and reduce unsupported or hallucinated concepts.

In [24]:
def get_class_comment(ontology_data, class_name):
    for c in ontology_data["classes"]:
        if c["name"] == class_name:
            return c["comment"]
    return ""


def summarized_ontology(ontology_data):
    schema = "SUMMARIZED ONTOLOGY SCHEMA\n\n"

    schema += "1. Corner Case Categories:\n"
    for c in ontology_data["classes"]:
        if c["name"] in ["StateAnomaly", "BehaviorAnomaly", "EvidenceBasedAnomaly"]:
            comment = c["comment"]
            schema += f"- {c['name']}: {comment}\n" if comment else f"- {c['name']}\n"

    schema += "\n2. Main Scene Concepts:\n"
    important_classes = [
        "Scene", "Actor", "Vehicle", "Pedestrian", "Obstacle",
        "RoadObject", "TrafficSign", "TrafficSignal",
        "WeatherCondition", "EvidenceIndicator", "Risk"
    ]

    for c in ontology_data["classes"]:
        if c["name"] in important_classes:
            comment = c["comment"]
            schema += f"- {c['name']}: {comment}\n" if comment else f"- {c['name']}\n"

    schema += "\n3. Risk Concepts:\n"
    for s in ontology_data["subclass_relations"]:
        if s["parent"] == "Risk":
            comment = get_class_comment(ontology_data, s["child"])
            schema += f"- {s['child']}: {comment}\n" if comment else f"- {s['child']}\n"

    schema += "\n4. Evidence Indicator Concepts:\n"
    for s in ontology_data["subclass_relations"]:
        if s["parent"] == "EvidenceIndicator":
            comment = get_class_comment(ontology_data, s["child"])
            schema += f"- {s['child']}: {comment}\n" if comment else f"- {s['child']}\n"

    schema += "\n5. Object / Road Concepts:\n"
    for s in ontology_data["subclass_relations"]:
        if s["parent"] in ["Vehicle", "Obstacle", "RoadObject", "WeatherCondition"]:
            comment = get_class_comment(ontology_data, s["child"])
            if comment:
                schema += f"- {s['child']} is a type of {s['parent']}: {comment}\n"
            else:
                schema += f"- {s['child']} is a type of {s['parent']}\n"

    schema += "\n6. Allowed Object Properties:\n"
    for p in ontology_data["object_properties"]:
        schema += f"- {p['name']}: {p['domain']} → {p['range']}. {p['comment']}\n"

    schema += "\n7. Allowed Datatype Properties:\n"
    for p in ontology_data["datatype_properties"]:
        schema += f"- {p['name']}: {p['domain']} → {p['range']}. {p['comment']}\n"

    return schema

def build_allowed_classes(ontology_data):
    text = "ALLOWED CLASSES:\n"
    for c in ontology_data["classes"]:
        text += f"- {c['name']}\n"
    return text


def build_allowed_relationships(ontology_data):
    text = "ALLOWED RELATIONSHIPS / OBJECT PROPERTIES:\n"
    for p in ontology_data["object_properties"]:
        text += f"- {p['name']}: {p['domain']} → {p['range']}\n"
    return text


def build_class_descriptions(ontology_data):
    text = "CLASS DESCRIPTIONS FROM ONTOLOGY COMMENTS:\n"
    for c in ontology_data["classes"]:
        if c["comment"]:
            text += f"- {c['name']}: {c['comment']}\n"
    return text

In [25]:
ontology_summary = summarized_ontology(ontology_data)
allowed_classes = build_allowed_classes(ontology_data)
allowed_relationships = build_allowed_relationships(ontology_data)
class_descriptions = build_class_descriptions(ontology_data)

In [26]:
print("\n===== ONTOLOGY SUMMARY =====\n")
print(ontology_summary)

print("\n===== ALLOWED CLASSES =====\n")
print(allowed_classes)

print("\n===== ALLOWED RELATIONSHIPS =====\n")
print(allowed_relationships)

print("\n===== CLASS DESCRIPTIONS =====\n")
print(class_descriptions)


===== ONTOLOGY SUMMARY =====

SUMMARIZED ONTOLOGY SCHEMA

1. Corner Case Categories:
- BehaviorAnomaly: A situation where an actor behaves in an unexpected or unsafe way.
- EvidenceBasedAnomaly: A scenario where observable evidence indicates a potential future risk.
- StateAnomaly: A mismatch between the appearance of an object and its expected meaning.

2. Main Scene Concepts:
- Actor: A dynamic entity in the scene such as a vehicle or pedestrian.
- EvidenceIndicator: A signal or clue that suggests a potential upcoming risk.
- Obstacle: A physical object present on the road that may obstruct the vehicle’s path.
- Pedestrian
- Risk: A potential hazard that may lead to unsafe situations.
- RoadObject: An object present in the driving environment.
- Scene: The type of driving environment where the scenario occurs.
- TrafficSign: A road sign that provides regulatory, warning, or informational instructions to drivers.
- TrafficSignal: A signaling device that controls traffic flow using li

#Ontology-Based Prompt Construction
Builds the final ontology-guided prompt by inserting the ontology summary, allowed classes, allowed relationships, and class descriptions. This helps the model analyze images using structured semantic knowledge instead of only free form text.

It also defines the exact tasks the model must complete, including entity detection, relationship identification, corner case classification, RDF generation, risk naming, and severity scoring. The prompt also enforces a fixed RDF structure so risks can be parsed and evaluated consistently

In [27]:
ontology_prompt = f"""
You are an autonomous driving safety analyst.

Your task is to analyze the following scene images and produce a semantic interpretation of the environment. The images represent a potential corner case scenario.

Use the ontology information below to guide your reasoning and ensure semantic consistency.

1. SUMMARIZED ONTOLOGY SCHEMA
{ontology_summary}

2. ALLOWED CLASSES
{allowed_classes}

3. ALLOWED RELATIONSHIPS
{allowed_relationships}

4. CLASS DESCRIPTIONS
{class_descriptions}

TASK

1. Identify the main entities in the scene:
   (vehicles, pedestrians, cyclists, road infrastructure, obstacles, weather conditions)
   - Prefer ontology-defined classes

2. Identify relationships between entities:
   (approaching, following, crossing, occluding, near)
   - Prefer ontology-defined relationships

3. Classify the scenario into ONE corner case type:
   - StateAnomaly
   - BehaviorAnomaly
   - EvidenceBasedAnomaly

4. Generate RDF triples describing the scene:
   - Use Turtle format
   - Ensure consistency with ontology classes and relationships

5. Identify potential safety risks:
   - Use ontology-defined risk concepts where possible
   - If the exact risk is not available, map to the closest ontology risk class
   - Describe the specific risk clearly based on what is visible in the images
   - Do NOT invent risks that are not supported by the images
   - Clearly NAME each risk

6. Assign a severity score between 0 and 1:
   - 0 = very low risk
   - 1 = very high risk

CRITICAL RDF REQUIREMENT

For EVERY risk in the Turtle output, you MUST use this exact structure:

:Risk_1 a :SomeRiskType ;
    :hasDescription "Clear image-based description of the risk." ;
    :hasSeverityScore "0.85"^^xsd:float .

Rules:
- Every risk MUST have:
   - rdf:type (or 'a') with a valid Risk class
   - hasDescription (MANDATORY)
   - hasSeverityScore (MANDATORY)
- Do NOT create risks without descriptions
- Each risk must be a separate RDF resource

IMPORTANT


- Use ontology concepts as guidance, not strict limitation
- Only include risks that are visually supported by the images
- Avoid hallucinated or unsupported entities, relationships, or risks
- Maintain consistency across entities, RDF triples, and risks

OUTPUT FORMAT

Scene Description in RDF Triples (Turtle Format)

Identified Risks

Risk 1:
Name:
Description:
Severity Score:

Risk 2:
Name:
Description:
Severity Score:
"""

#Model Execution
This pipeline follows the same structure as the previous model execution pipeline, with the main difference being the use of an ontology-based prompt instead of a baseline prompt.

The model processes batched image inputs, generates outputs, and records runtime in a similar way. However, the ontology-guided prompt ensures more structured and semantically consistent outputs.

The generated responses are parsed to extract RDF (Turtle) content, and risks are identified using RDF-based extraction with a fallback to regex when needed. The results are stored in JSON and TTL formats along with relevant metadata.

Overall, the pipeline remains the same, with the key improvement being the integration of ontology-based prompting for better consistency and evaluation.

#Gemini

In [32]:
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [33]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Say: Gemini is connected."
)

print(response.text)

Gemini is connected.


In [34]:
model_name = "gemini-3-flash-preview"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

all_results = []

for scenario_data in all_scenarios_data:
    scenario_id = safe_name(
        f'{scenario_data["condition"]}_{scenario_data["anomaly"]}_{scenario_data["scenario"]}'
    )

    print(f"\nProcessing scenario: {scenario_id}")

    for batch_index, batch_paths in enumerate(scenario_data["batches"], start=1):
        print(f"  Batch {batch_index}/{len(scenario_data['batches'])}")

        batch_images = [Image.open(img_path) for img_path in batch_paths]

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                start_time = time.time()

                response = client.models.generate_content(
                    model=model_name,
                    contents=[ontology_prompt, *batch_images]
                )

                end_time = time.time()
                runtime = end_time - start_time

                raw_output = response.text

                ttl_output = extract_ttl(raw_output)

                rdf_risks = extract_risks_from_rdf(ttl_output)

                if rdf_risks:
                    parsed_risks = rdf_risks
                    extraction_source = "RDF"
                else:

                    parsed_risks = extract_risks(raw_output)
                    extraction_source = "REGEX_FALLBACK"

                output_data = {
                    "model": model_name,
                    "prompt_type": "ontology_guided",
                    "extraction_source": extraction_source,
                    "scenario": {
                        "model": model_name,
                        "condition": scenario_data["condition"],
                        "anomaly": scenario_data["anomaly"],
                        "scenario": scenario_data["scenario"]
                    },
                    "batch_size": batch_size,
                    "batch_number": batch_index,
                    "batch_paths": batch_paths,
                    "runtime_seconds": runtime,
                    "raw_output": raw_output,
                    "ttl_output": ttl_output,
                    "parsed_risks": parsed_risks
                }

                json_file = os.path.join(
                    output,
                    f"{model_name}_ontology_bs{batch_size}_{scenario_id}_batch{batch_index}_output.json"
                )

                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(output_data, f, indent=4)

                ttl_file = os.path.join(
                    output,
                    f"{model_name}_ontology_bs{batch_size}_{scenario_id}_batch{batch_index}_output.ttl"
                )

                with open(ttl_file, "w", encoding="utf-8") as f:
                    f.write(ttl_output if ttl_output else "")

                all_results.append(output_data)

                print(f"    Runtime: {runtime:.2f} sec")
                print(f"    Extraction source: {extraction_source}")
                print(f"    Risks extracted: {len(parsed_risks)}")
                print(f"    Saved JSON: {json_file}")
                print(f"    Saved TTL: {ttl_file}")

                success = True
                time.sleep(2)

            except Exception as e:
                retries -= 1
                print(f"    Error: {e}")
                print("    Retrying...")
                time.sleep(5)

        if not success:
            print(f"    Failed batch {batch_index}, skipping...")


Processing scenario: NightRainy_StateAnomaly_StaticHighwayAd4
  Batch 1/2
    Runtime: 21.49 sec
    Extraction source: RDF
    Risks extracted: 4
    Saved JSON: /content/drive/MyDrive/semantic web project 2/ran/gemini-3-flash-preview_ontology_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
    Saved TTL: /content/drive/MyDrive/semantic web project 2/ran/gemini-3-flash-preview_ontology_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.ttl
  Batch 2/2
    Runtime: 19.08 sec
    Extraction source: RDF
    Risks extracted: 2
    Saved JSON: /content/drive/MyDrive/semantic web project 2/ran/gemini-3-flash-preview_ontology_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.json
    Saved TTL: /content/drive/MyDrive/semantic web project 2/ran/gemini-3-flash-preview_ontology_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.ttl

Processing scenario: DayClear_EvidenceBasedAnomaly_FallenTree
  Batch 1/3
    Runtime: 24.14 sec
    Extraction source: RD

In [35]:
output_txt_path = f"gemini3_ontology_report_from_ttl_batch{batch_size}.txt"

fixed_summary = []
target_model = "gemini-3-flash-preview"

with open(output_txt_path, "w", encoding="utf-8") as txt_file:
    for file_name in os.listdir(output):

        if not file_name.endswith(".ttl"):
            continue
        if not file_name.startswith(target_model):
            continue
        if f"_bs{batch_size}_" not in file_name:
            continue

        ttl_file_path = os.path.join(output, file_name)

        with open(ttl_file_path, "r", encoding="utf-8") as f:
            ttl_output = f.read()

        parsed_risks = extract_risks_from_rdf(ttl_output)
        extraction_source = "TTL_FILE_RDF"

        json_file_name = file_name.replace(".ttl", ".json")
        json_file_path = os.path.join(output, json_file_name)

        runtime = 0
        batch_paths = []
        batch_number = None
        scenario = {
            "condition": "UNKNOWN",
            "anomaly": "UNKNOWN",
            "scenario": "UNKNOWN"
        }

        if os.path.exists(json_file_path):
            with open(json_file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            scenario = data["scenario"]
            runtime = data.get("runtime_seconds", 0)
            batch_number = data.get("batch_number")
            batch_paths = data.get("batch_paths", [])

        header = f"\n{'-'*70}\n"
        file_info = f"TTL File: {file_name}\n"
        scenario_info = f"Scenario: {scenario['condition']} | {scenario['anomaly']} | {scenario['scenario']}\n"
        batch_info = f"Batch: {batch_number}\n"
        runtime_info = f"Runtime: {runtime}\n"
        source_info = f"Extraction source: {extraction_source}\n"
        size_info = f"Input size: {len(batch_paths)}\nOutput size: {len(parsed_risks)}\n"

        print(header + file_info + scenario_info + batch_info + runtime_info + source_info + size_info, end="")
        txt_file.write(header + file_info + scenario_info + batch_info + runtime_info + source_info + size_info)

        for i, risk in enumerate(parsed_risks, start=1):
            risk_text = (
                f"  {i}. {risk.get('description', '')}\n"
                f"     Risk Type: {risk.get('risk_type', '')}\n"
                f"     Severity: {risk.get('severity', None)}\n"
            )
            print(risk_text, end="")
            txt_file.write(risk_text)

        for risk in parsed_risks:
            fixed_summary.append({
                "condition": scenario["condition"],
                "anomaly": scenario["anomaly"],
                "scenario": scenario["scenario"],
                "batch": batch_number,
                "risk_type": risk.get("risk_type", ""),
                "risk_description": risk.get("description", ""),
                "severity": risk.get("severity", None),
                "runtime": runtime,
                "input_size": len(batch_paths),
                "output_size": len(parsed_risks),
                "extraction_source": extraction_source
            })

try:
    from google.colab import files
    files.download(output_txt_path)
except ImportError:
    print(f"\nDone! Your file is saved at: {os.path.abspath(output_txt_path)}")


----------------------------------------------------------------------
TTL File: gemini-3-flash-preview_ontology_bs5_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.ttl
Scenario: NightRainy | StateAnomaly | StaticHighwayAd4
Batch: 1
Runtime: 21.48732614517212
Extraction source: TTL_FILE_RDF
Input size: 5
Output size: 4
  1. The ambiguity of the signal states at an intersection increases the risk of a side-impact or head-on collision with other actors who may be navigating the intersection under different assumptions.
     Risk Type: CollisionRisk
     Severity: 0.8
  2. The asphalt surface is wet and reflective due to ongoing rain, which reduces tire traction and increases the distance required to stop safely.
     Risk Type: LowFrictionRisk
     Severity: 0.75
  3. Visibility is degraded by the combination of night-time low lighting, rain, and environmental glare reflected off the wet road surface.
     Risk Type: ReducedVisibilityRisk
     Severity: 0.65
  4. The presence of 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#ChatGPT-5.4-mini

In [ ]:
!pip -q install -U openai
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

Enter your OpenAI API key: ··········


In [ ]:
import base64
import mimetypes

def encode_image_to_data_url(image_path):
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"


In [ ]:
def call_chatgpt_batch(image_paths, prompt, model_name="gpt-5.4-mini"):
    content = [{"type": "input_text", "text": prompt}]

    for img_path in image_paths:
        data_url = encode_image_to_data_url(img_path)

        content.append({
            "type": "input_image",
            "image_url": data_url
        })

    response = openai_client.responses.create(
        model=model_name,
        input=[{
            "role": "user",
            "content": content
        }]
    )

    return response.output_text

In [ ]:
chatgpt_model_name = "gpt-5.4-mini"
all_chatgpt_results = []

for scenario_data in all_scenarios_data:
    scenario_id = safe_name(
        f'{scenario_data["condition"]}_{scenario_data["anomaly"]}_{scenario_data["scenario"]}'
    )

    print(f"\nProcessing ChatGPT ontology scenario: {scenario_id}")

    for batch_index, batch_paths in enumerate(scenario_data["batches"], start=1):
        print(f" Batch {batch_index}/{len(scenario_data['batches'])}")

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                start_time = time.time()

                raw_output = call_chatgpt_batch(
                    image_paths=batch_paths,
                    prompt=ontology_prompt,
                    model_name=chatgpt_model_name
                )

                end_time = time.time()
                runtime = end_time - start_time

                ttl_output = extract_ttl(raw_output)

                rdf_risks = extract_risks_from_rdf(ttl_output)

                if rdf_risks:
                    parsed_risks = rdf_risks
                    extraction_source = "RDF"
                else:
                    parsed_risks = extract_risks(raw_output)
                    extraction_source = "REGEX_FALLBACK"

                output_data = {
                    "model": chatgpt_model_name,
                    "prompt_type": "ontology_guided",
                    "extraction_source": extraction_source,
                    "scenario": {
                        "model": chatgpt_model_name,
                        "condition": scenario_data["condition"],
                        "anomaly": scenario_data["anomaly"],
                        "scenario": scenario_data["scenario"]
                    },
                    "batch_size": batch_size,
                    "batch_number": batch_index,
                    "batch_paths": batch_paths,
                    "runtime_seconds": runtime,
                    "raw_output": raw_output,
                    "ttl_output": ttl_output,
                    "parsed_risks": parsed_risks
                }

                json_file = os.path.join(
                    output,
                    f"{chatgpt_model_name}_ontology_bs{batch_size}_{scenario_id}_batch{batch_index}_output.json"
                )

                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(output_data, f, indent=4)

                ttl_file = os.path.join(
                    output,
                    f"{chatgpt_model_name}_ontology_bs{batch_size}_{scenario_id}_batch{batch_index}_output.ttl"
                )

                with open(ttl_file, "w", encoding="utf-8") as f:
                    f.write(ttl_output if ttl_output else "")

                all_chatgpt_results.append(output_data)

                print(f" Runtime: {runtime:.2f} sec")
                print(f" Extraction source: {extraction_source}")
                print(f" Risks extracted: {len(parsed_risks)}")
                print(f" Saved JSON: {json_file}")
                print(f" Saved TTL: {ttl_file}")

                success = True
                time.sleep(2)

            except Exception as e:
                retries -= 1
                print(f" Error: {e}")
                print(" Retrying...")
                time.sleep(5)

        if not success:
            print(f" Failed batch {batch_index}, skipping...")


Processing ChatGPT ontology scenario: NightRainy_StateAnomaly_StaticHighwayAd4
 Batch 1/10
 Runtime: 3.84 sec
 Extraction source: RDF
 Risks extracted: 2
 Saved JSON: /content/drive/MyDrive/semantic web project 2/model output/gpt-5.4-mini_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
 Saved TTL: /content/drive/MyDrive/semantic web project 2/model output/gpt-5.4-mini_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.ttl
 Batch 2/10
 Runtime: 2.93 sec
 Extraction source: RDF
 Risks extracted: 2
 Saved JSON: /content/drive/MyDrive/semantic web project 2/model output/gpt-5.4-mini_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.json
 Saved TTL: /content/drive/MyDrive/semantic web project 2/model output/gpt-5.4-mini_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.ttl
 Batch 3/10
 Runtime: 3.19 sec
 Extraction source: RDF
 Risks extracted: 2
 Saved JSON: /content/drive/MyDrive/semantic web project 2/model ou

In [ ]:
output_txt_path = f"chatgpt_ontology_report_batch{batch_size}.txt"

fixed_summary = []
target_model = "gpt-5.4-mini"

with open(output_txt_path, "w", encoding="utf-8") as txt_file:
    for file_name in sorted(os.listdir(output)):
        if not file_name.endswith(".json"):
            continue
        if not file_name.startswith(target_model):
            continue
        if f"_ontology_bs{batch_size}_" not in file_name:
            continue

        file_path = os.path.join(output, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        scenario = data["scenario"]
        runtime = data.get("runtime_seconds", 0)
        batch_number = data.get("batch_number")
        batch_paths = data.get("batch_paths", [])
        raw_output = data.get("raw_output", "")

        parsed_risks = data.get("parsed_risks", [])
        extraction_source = data.get("extraction_source", "UNKNOWN")

        if not parsed_risks:
            ttl_output = data.get("ttl_output", "")

            if not ttl_output:
                ttl_output = extract_ttl(raw_output)

            rdf_risks = extract_risks_from_rdf(ttl_output)

            if rdf_risks:
                parsed_risks = rdf_risks
                extraction_source = "RDF_FROM_TTL"
            else:
                parsed_risks = extract_risks(raw_output)
                extraction_source = "REGEX_FALLBACK"

        header = f"\n{'-'*70}\n"
        file_info = f"File: {file_name}\n"
        scenario_info = f"Scenario: {scenario['condition']} | {scenario['anomaly']} | {scenario['scenario']}\n"
        batch_info = f"Batch: {batch_number}\n"
        runtime_info = f"Runtime: {runtime}\n"
        source_info = f"Extraction source: {extraction_source}\n"
        size_info = f"Input size: {len(batch_paths)}\nOutput size: {len(parsed_risks)}\n"

        print(header + file_info + scenario_info + batch_info + runtime_info + source_info + size_info, end="")
        txt_file.write(header + file_info + scenario_info + batch_info + runtime_info + source_info + size_info)

        for i, risk in enumerate(parsed_risks, start=1):
            risk_text = (
                f"  {i}. {risk.get('description', '')}\n"
                f"     Risk Type: {risk.get('risk_type', '')}\n"
                f"     Severity: {risk.get('severity', None)}\n"
            )
            print(risk_text, end="")
            txt_file.write(risk_text)

        for risk in parsed_risks:
            fixed_summary.append({
                "method": "Ontology-Grounded",
                "model": chatgpt_model_name,
                "condition": scenario["condition"],
                "anomaly": scenario["anomaly"],
                "scenario": scenario["scenario"],
                "batch": batch_number,
                "risk_type": risk.get("risk_type", ""),
                "risk_description": risk.get("description", ""),
                "severity": risk.get("severity", None),
                "runtime": runtime,
                "input_size": len(batch_paths),
                "output_size": len(parsed_risks),
                "extraction_source": extraction_source
            })

try:
    from google.colab import files
    files.download(output_txt_path)
except ImportError:
    print(f"\nfile is saved at: {os.path.abspath(output_txt_path)}")


----------------------------------------------------------------------
File: gpt-5.4-mini_ontology_bs1_DayClear_BehaviorAnomaly_BusObscuresCar_batch10_output.json
Scenario: DayClear | BehaviorAnomaly | BusObscuresCar
Batch: 10
Runtime: 4.178705453872681
Extraction source: RDF
Input size: 1
Output size: 2
  1. The scene shows an approaching signalized intersection with multiple vehicles and lane guidance that may require careful compliance with traffic rules and lane positioning.
     Risk Type: TrafficViolationRisk
     Severity: 0.52
  2. A vehicle is positioned very close to the ego vehicle on the right, creating a side-swipe or merging collision risk.
     Risk Type: CollisionRisk
     Severity: 0.78

----------------------------------------------------------------------
File: gpt-5.4-mini_ontology_bs1_DayClear_BehaviorAnomaly_BusObscuresCar_batch1_output.json
Scenario: DayClear | BehaviorAnomaly | BusObscuresCar
Batch: 1
Runtime: 4.1849284172058105
Extraction source: RDF
Input siz

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#LLaMA

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.9 MB/s eta 0:00:00


In [ ]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
def encode_image_for_groq(path):
    mime_type, _ = mimetypes.guess_type(path)
    if mime_type is None:
        mime_type = "image/png"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"

In [ ]:

def call_llama_batch(image_paths, prompt, model_name):
    content = [{"type": "text", "text": prompt}]

    for img_path in image_paths:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": encode_image_for_groq(img_path)
            }
        })

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": content}],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
llama_model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
safe_model_name = llama_model_name.replace("/", "-")

all_llama_results = []

for scenario_data in all_scenarios_data:
    scenario_id = safe_name(
        f'{scenario_data["condition"]}_{scenario_data["anomaly"]}_{scenario_data["scenario"]}'
    )

    print(f"\nProcessing LLaMA ontology scenario: {scenario_id}")

    for batch_index, batch_paths in enumerate(scenario_data["batches"], start=1):
        print(f" Batch {batch_index}/{len(scenario_data['batches'])}")

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                start_time = time.time()

                raw_output = call_llama_batch(
                    image_paths=batch_paths,
                    prompt=ontology_prompt,
                    model_name=llama_model_name
                )

                end_time = time.time()
                runtime = end_time - start_time

                ttl_output = extract_ttl(raw_output)

                rdf_risks = extract_risks_from_rdf(ttl_output)

                if rdf_risks:
                    parsed_risks = rdf_risks
                    extraction_source = "RDF"
                else:
                    parsed_risks = extract_risks(raw_output)
                    extraction_source = "REGEX_FALLBACK"

                output_data = {
                    "model": llama_model_name,
                    "prompt_type": "ontology_guided",
                    "extraction_source": extraction_source,
                    "scenario": {
                        "model": llama_model_name,
                        "condition": scenario_data["condition"],
                        "anomaly": scenario_data["anomaly"],
                        "scenario": scenario_data["scenario"]
                    },
                    "batch_size": batch_size,
                    "batch_number": batch_index,
                    "batch_paths": batch_paths,
                    "runtime_seconds": runtime,
                    "raw_output": raw_output,
                    "ttl_output": ttl_output,
                    "parsed_risks": parsed_risks
                }

                json_file = os.path.join(
                    output,
                    f"{safe_model_name}_ontology_bs{batch_size}_{scenario_id}_batch{batch_index}_output.json"
                )

                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(output_data, f, indent=4)

                ttl_file = os.path.join(
                    output,
                    f"{safe_model_name}_ontology_bs{batch_size}_{scenario_id}_batch{batch_index}_output.ttl"
                )

                with open(ttl_file, "w", encoding="utf-8") as f:
                    f.write(ttl_output if ttl_output else "")

                all_llama_results.append(output_data)

                print(f" Runtime: {runtime:.2f} sec")
                print(f" Extraction source: {extraction_source}")
                print(f" Risks extracted: {len(parsed_risks)}")
                print(f" Saved JSON: {json_file}")
                print(f" Saved TTL: {ttl_file}")

                success = True
                time.sleep(2)

            except Exception as e:
                retries -= 1
                print(f" Error: {e}")
                print(" Retrying...")
                time.sleep(5)

        if not success:
            print(f" Failed batch {batch_index}, skipping...")


Processing LLaMA ontology scenario: NightRainy_StateAnomaly_StaticHighwayAd4
 Batch 1/10
 Runtime: 2.63 sec
 Extraction source: RDF
 Risks extracted: 2
 Saved JSON: /content/drive/MyDrive/semantic web project 2/model output/meta-llama-llama-4-scout-17b-16e-instruct_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.json
 Saved TTL: /content/drive/MyDrive/semantic web project 2/model output/meta-llama-llama-4-scout-17b-16e-instruct_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch1_output.ttl
 Batch 2/10
 Runtime: 2.19 sec
 Extraction source: RDF
 Risks extracted: 2
 Saved JSON: /content/drive/MyDrive/semantic web project 2/model output/meta-llama-llama-4-scout-17b-16e-instruct_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.json
 Saved TTL: /content/drive/MyDrive/semantic web project 2/model output/meta-llama-llama-4-scout-17b-16e-instruct_ontology_bs1_NightRainy_StateAnomaly_StaticHighwayAd4_batch2_output.ttl
 Batch 3/10
 Runtime: 1.86 s

In [ ]:
output_txt_path = f"llama_groq_ontology_report_batch{batch_size}.txt"

fixed_summary = []
target_model = "meta-llama-llama-4-scout-17b-16e-instruct"

with open(output_txt_path, "w", encoding="utf-8") as txt_file:
    for file_name in sorted(os.listdir(output)):
        if not file_name.endswith(".json"):
            continue
        if not file_name.startswith(target_model):
            continue
        if f"_ontology_bs{batch_size}_" not in file_name:
            continue

        file_path = os.path.join(output, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        scenario = data["scenario"]
        runtime = data.get("runtime_seconds", 0)
        batch_number = data.get("batch_number")
        batch_paths = data.get("batch_paths", [])
        raw_output = data.get("raw_output", "")

        parsed_risks = data.get("parsed_risks", [])
        extraction_source = data.get("extraction_source", "UNKNOWN")

        if not parsed_risks:
            ttl_output = data.get("ttl_output", "")

            if not ttl_output:
                ttl_output = extract_ttl(raw_output)

            rdf_risks = extract_risks_from_rdf(ttl_output)

            if rdf_risks:
                parsed_risks = rdf_risks
                extraction_source = "RDF_FROM_TTL"
            else:
                parsed_risks = extract_risks(raw_output)
                extraction_source = "REGEX_FALLBACK"

        header = f"\n{'-'*70}\n"
        file_info = f"File: {file_name}\n"
        scenario_info = f"Scenario: {scenario['condition']} | {scenario['anomaly']} | {scenario['scenario']}\n"
        batch_info = f"Batch: {batch_number}\n"
        runtime_info = f"Runtime: {runtime}\n"
        source_info = f"Extraction source: {extraction_source}\n"
        size_info = f"Input size: {len(batch_paths)}\nOutput size: {len(parsed_risks)}\n"

        print(header + file_info + scenario_info + batch_info + runtime_info + source_info + size_info, end="")
        txt_file.write(header + file_info + scenario_info + batch_info + runtime_info + source_info + size_info)

        for i, risk in enumerate(parsed_risks, start=1):
            risk_text = (
                f"  {i}. {risk.get('description', '')}\n"
                f"     Risk Type: {risk.get('risk_type', '')}\n"
                f"     Severity: {risk.get('severity', None)}\n"
            )
            print(risk_text, end="")
            txt_file.write(risk_text)

        for risk in parsed_risks:
            fixed_summary.append({
                "method": "Ontology-Grounded",
                "model": llama_model_name,
                "condition": scenario["condition"],
                "anomaly": scenario["anomaly"],
                "scenario": scenario["scenario"],
                "batch": batch_number,
                "risk_type": risk.get("risk_type", ""),
                "risk_description": risk.get("description", ""),
                "severity": risk.get("severity", None),
                "runtime": runtime,
                "input_size": len(batch_paths),
                "output_size": len(parsed_risks),
                "extraction_source": extraction_source
            })

try:
    from google.colab import files
    files.download(output_txt_path)
except ImportError:
    print(f"\nfile is saved at: {os.path.abspath(output_txt_path)}")


----------------------------------------------------------------------
File: meta-llama-llama-4-scout-17b-16e-instruct_ontology_bs1_DayClear_BehaviorAnomaly_BusObscuresCar_batch10_output.json
Scenario: DayClear | BehaviorAnomaly | BusObscuresCar
Batch: 10
Runtime: 2.262892246246338
Extraction source: RDF
Input size: 1
Output size: 2
  1. Risk of lane intrusion if the black car changes lanes unexpectedly
     Risk Type: LaneIntrusionRisk
     Severity: 0.4
  2. Potential collision risk with the black car if it stops suddenly
     Risk Type: CollisionRisk
     Severity: 0.6

----------------------------------------------------------------------
File: meta-llama-llama-4-scout-17b-16e-instruct_ontology_bs1_DayClear_BehaviorAnomaly_BusObscuresCar_batch1_output.json
Scenario: DayClear | BehaviorAnomaly | BusObscuresCar
Batch: 1
Runtime: 2.0646471977233887
Extraction source: RDF
Input size: 1
Output size: 2
  1. Potential collision risk with vehicles in the vicinity.
     Risk Type: Collisio

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Ground Truth
The ground truth defines the expected risks for each scenario using a predefined dictionary.
These risks are manually selected based on scenario conditions and used as a reference for evaluation.

In [28]:
ground_truth = {
    "NightRainy_StateAnomaly_StaticHighwayAd4": [
        "ReducedVisibilityRisk",
        "LowFrictionRisk",
        "LaneIntrusionRisk",
        "TrafficViolationRisk"
    ],

    "DayClear_EvidenceBasedAnomaly_FallenTree": [
        "ObstructionRisk",
        "OcclusionRisk",
        "TrafficViolationRisk",
        "CollisionRisk"
    ],

    "DayFoggy_EvidenceBasedAnomaly_FootballFollowedByBoy": [
        "ReducedVisibilityRisk",
        "OcclusionRisk",
        "SuddenObstacleRisk",
        "PedestrianCollisionRisk",
        "UnpredictableBehaviorRisk"
    ],

    "DayFoggy_BehaviorAnomaly_CarOppositeDirectionHIT": [
        "ReducedVisibilityRisk",
        "LaneIntrusionRisk",
        "CollisionRisk"
    ],

    "DayClear_BehaviorAnomaly_BusObscuresCar": [
        "OcclusionRisk",
        "UnpredictableBehaviorRisk",
        "LaneIntrusionRisk",
        "CollisionRisk"
    ]
}

This block normalizes risk names to ensure consistency between predicted and ground truth risks.

In [36]:
def normalize_risk_name(name):
    if not name:
        return ""

    name = name.strip()

    fixes = {
        "SuddenOstacleRisk": "SuddenObstacleRisk",
        "BehaviourAnomaly": "BehaviorAnomaly"
    }

    return fixes.get(name, name)

#Model Evaluation Metrics
It computes evaluation metrics including TP, FP, FN, precision, recall, F1 score, and hallucination rate by comparing predicted risks with ground truth.
These metrics are used to evaluate model performance across scenarios.

In [37]:
def compute_metrics(predicted_risks, ground_truth_risks):
    predicted = list(set([
        normalize_risk_name(r["risk_type"])
        for r in predicted_risks
        if r.get("risk_type")
    ]))

    truth = list(set([
        normalize_risk_name(r)
        for r in ground_truth_risks
    ]))

    tp = len(set(predicted) & set(truth))
    fp = len(set(predicted) - set(truth))
    fn = len(set(truth) - set(predicted))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
    hallucination_rate = fp / len(predicted) if len(predicted) > 0 else 0

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Hallucination Rate": hallucination_rate
    }

#Risk Grouping and Organization
This block groups predicted risks based on scenario, condition, anomaly type, and batch.
It organizes the data into structured groups for easier comparison with ground truth.
This grouping is required to compute evaluation metrics at the scenario and batch level.

In [38]:
from collections import defaultdict
import pandas as pd

grouped = defaultdict(list)

for row in fixed_summary:
    scenario_id = safe_name(
        f"{row['condition']}_{row['anomaly']}_{row['scenario']}"
    )

    key = (
        scenario_id,
        row["condition"],
        row["anomaly"],
        row["scenario"],
        row["batch"]
    )

    grouped[key].append({
        "risk_type": row["risk_type"],
        "description": row["risk_description"],
        "severity": row["severity"]
    })

#Evaluation Metrics Aggregation

This block computes evaluation metrics for each grouped scenario and batch using the ground truth.
It stores TP, FP, FN, precision, recall, F1 score, and hallucination rate in a structured format.
The results are then converted into a DataFrame for analysis and reporting.

In [39]:
metric_rows = []

for key, predicted_risks in grouped.items():
    scenario_id, condition, anomaly, scenario, batch = key

    gt = ground_truth.get(scenario_id, [])

    metrics = compute_metrics(predicted_risks, gt)

    metric_rows.append({
        "Model": "llama_groq_ontology",
        "Scenario": scenario,
        "Weather Condition": condition,
        "Anomaly": anomaly,
        "Batch": batch,
        "TP": metrics["TP"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "Precision": round(metrics["Precision"], 3),
        "Recall": round(metrics["Recall"], 3),
        "F1 Score": round(metrics["F1"], 3),
        "Hallucination Rate": round(metrics["Hallucination Rate"], 3)
    })

metrics_df = pd.DataFrame(metric_rows)
metrics_df

,Model,Scenario,Weather Condition,Anomaly,Batch,TP,FP,FN,Precision,Recall,F1 Score,Hallucination Rate
0,llama_groq_ontology,StaticHighwayAd4,NightRainy,StateAnomaly,1,3,1,1,0.750,0.750,0.750,0.250
1,llama_groq_ontology,StaticHighwayAd4,NightRainy,StateAnomaly,2,2,0,2,1.000,0.500,0.667,0.000
2,llama_groq_ontology,FallenTree,DayClear,EvidenceBasedAnomaly,1,2,1,2,0.667,0.500,0.571,0.333
3,llama_groq_ontology,FallenTree,DayClear,EvidenceBasedAnomaly,2,2,1,2,0.667,0.500,0.571,0.333
4,llama_groq_ontology,FallenTree,DayClear,EvidenceBasedAnomaly,3,3,0,1,1.000,0.750,0.857,0.000
5,llama_groq_ontology,FootballFollowedByBoy,DayFoggy,EvidenceBasedAnomaly,1,3,0,2,1.000,0.600,0.750,0.000
6,llama_groq_ontology,FootballFollowedByBoy,DayFoggy,EvidenceBasedAnomaly,2,2,1,3,0.667,0.400,0.500,0.333
7,llama_groq_ontology,FootballFollowedByBoy,DayFoggy,EvidenceBasedAnomaly,3,3,0,2,1.000,0.600,0.750,0.000
8,llama_groq_ontology,CarOppositeDirectionHIT,DayFoggy,BehaviorAnomaly,1,2,0,1,1.000,0.667,0.800,0.000
9,llama_groq_ontology,CarOppositeDirectionHIT,DayFoggy,BehaviorAnomaly,2,3,0,0,1.000,1.000,1.000,0.000


#Metrics Export and Download

This block saves the computed evaluation metrics as a CSV file.
It also downloads the file in Colab for easy access and use in reports.
If download is not supported, the file is still saved locally.

In [40]:
metrics_df.to_csv(f"llama_groq_ontology_metric_batch{batch_size}.csv", index=False)

try:
    from google.colab import files
    files.download(f"llama_groq_ontology_metric_batch{batch_size}.csv")
except ImportError:
    print("Saved metrics CSV.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Experimental Pipeline – Ontology-Based

Dataset → Scenario Selection → Batching → Ontology-Guided Model Inference → RDF Extraction → Risk Parsing → Metrics Evaluation → Reporting